<a href="https://colab.research.google.com/github/jenny4890/deepLearning/blob/main/transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torchvision
import matplotlib.pyplot as plt

if torch.backends.mps.is_available():
  my_device = torch.device('mps')
elif torch.cuda.is_available():
  my_device = torch.device('cuda')
else:
  my_device = torch.device('cpu')

print(my_device)

cuda


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406] , std=[0.229, 0.224, 0.225])
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size = 32, shuffle=True, num_workers=2)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(trainset, batch_size = 32, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:14<00:00, 12.1MB/s]


In [3]:
from torchvision import models
# 이 모델은 모든 parameter가 파인튜닝된다.
# 실제로 각 class별로 1K-2K정도 이미지를 모으고 모두 파인튜닝하는식으로 하면된다.
# 일부 Freeze한 방식도 있지만 그것보다는 위에 방법대로. 모드 파인튜닝. 이미지 확보에 전념.
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# 1. 모든 기존 파라미터의 미분(학습)을 비활성화
#for param in model.parameters():
#    param.requires_grad = False

model.fc = nn.Linear(in_features=512, out_features=10, bias=True)
#print(model)

criterion = nn.CrossEntropyLoss()
opimizer = optim.Adam(model.parameters(), lr=0.0001)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 207MB/s]


In [ ]:
import time

model.to(my_device)
num_epochs=100
for epoch in range(num_epochs):
  start_time = time.time()
  model.train()
  for batchidx, (data, label) in  enumerate(trainloader):
    data, label = data.to(my_device), label.to(my_device)
    scores = model(data)
    loss = criterion(scores,label)

    opimizer.zero_grad()
    loss.backward()
    opimizer.step()
  model.eval()
  valloss=0.0
  correct=0
  with torch.no_grad():
    for data, label in testloader:
      data, label = data.to(my_device), label.to(my_device)
      scores  = model(data)
      loss  = criterion(scores , label) #error 평균
      valloss +=loss.item()*data.size(0) # (N, C, H, W)중에 0번째 그러니까 batchsize.loss값을 배치만큼 곱한거
      #loss.item()는 torch.Tensor to 	float변환

      predicted = scores.argmax(dim=1) # dim=0 batch 방향, dim=1 가로 방향 class방향.
      #이거 이해 해야함! 만약 배치 사이즈가 3이고 클래스가 4개인 간단한 예시가 있다고 가정해 봅시다. output의 형태는 (3, 4)가 됩니다.
      '''
      (3, 4) 3이 0, 4가 1 dim=0 (세로 방향 / Batch 방향): dim=1 (가로 방향 / Class 방향):
      # [이미지1의 점수 4개], [이미지2의 점수 4개], [이미지3의 점수 4개]
      output = torch.tensor([
      [0.1, 0.2, 0.7, 0.0],  # 이미지 0 (정답 후보들)
      [0.1, 0.9, 0.0, 0.0],  # 이미지 1 (정답 후보들)
      [0.3, 0.3, 0.2, 0.2]   # 이미지 2 (정답 후보들)
      )
      '''
      correct += predicted.eq(label).sum().item() #item()는 torch.Tensor를float변환
    valloss  /= len(testloader.dataset)
    valacc = 100*correct/len(testloader.dataset)
    end_time=time.time()
    elapsedtime = end_time - start_time  # Compute the elapsed time
    print(f"Epoch [{epoch + 1}/{num_epochs}], Training Loss: {loss.item():.4f}, Validation Loss: {valloss:.4f}, Validation Accuracy: {valacc:.2f}%, Time: {elapsedtime:.2f}s")


Epoch [1/100], Training Loss: 0.2415, Validation Loss: 0.1080, Validation Accuracy: 96.60%, Time: 246.54s
Epoch [2/100], Training Loss: 0.0303, Validation Loss: 0.0580, Validation Accuracy: 98.08%, Time: 251.91s
Epoch [3/100], Training Loss: 0.0747, Validation Loss: 0.0346, Validation Accuracy: 99.00%, Time: 251.86s
Epoch [4/100], Training Loss: 0.0146, Validation Loss: 0.0204, Validation Accuracy: 99.37%, Time: 250.55s
Epoch [5/100], Training Loss: 0.0459, Validation Loss: 0.0215, Validation Accuracy: 99.32%, Time: 253.95s
Epoch [6/100], Training Loss: 0.0067, Validation Loss: 0.0166, Validation Accuracy: 99.49%, Time: 251.76s
Epoch [7/100], Training Loss: 0.0095, Validation Loss: 0.0170, Validation Accuracy: 99.46%, Time: 252.59s
Epoch [8/100], Training Loss: 0.0086, Validation Loss: 0.0227, Validation Accuracy: 99.23%, Time: 253.23s
